# Exploratory Data Analysis (EDA) - NHANES Demographics (`DEMO_L.xpt`)

This notebook performs a comprehensive exploratory and descriptive analysis of the **NHANES August 2021 - August 2023** Demographics dataset (`DEMO_L.xpt`).

### Objectives:
1. **Load Data**: Read `DEMO_L.xpt` from the Demographics data directory.
2. **Structure & Dimensions**: Inspect total rows, columns, memory usage, and data types.
3. **Feature Inventory**: Display all dataset features with data types and sample values.
4. **Missingness Analysis**: Calculate missing value counts, percentages, and identify incomplete features.
5. **Descriptive Statistics**: Summarize distributions (mean, std, median, IQR, min/max, skewness).
6. **Key Demographic Analysis**: Frequency distributions for age, gender, race/ethnicity, household size, and income ratio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.style.use('ggplot')

# Locate DEMO_L.xpt file dynamically
current_dir = Path.cwd()
xpt_matches = list(current_dir.rglob('DEMO_L.xpt'))

if xpt_matches:
    file_path = xpt_matches[0]
else:
    # Fallback to known relative path from mir-medical-intelligence root
    file_path = Path('DATASETS/data/raw/NHANES/NHANES August 2021 - Auguest 2023/Data, Documentation, Codebooks/Demographics Data - Continuous NHANES/DEMO_L.xpt')

print(f"[INFO] Loading dataset from: {file_path.resolve()}")
df = pd.read_sas(file_path)
print(f"[SUCCESS] Dataset successfully loaded into pandas DataFrame!")

## 1. High-Level Dataset Overview & Structure

In [ ]:
num_rows, num_cols = df.shape
memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print("="*60)
print(f"DATASET SUMMARY OVERVIEW")
print("="*60)
print(f"Total Number of Rows (Participants) : {num_rows:,}")
print(f"Total Number of Features (Columns)  : {num_cols}")
print(f"Total Memory Usage                 : {memory_mb:.2f} MB")
print("="*60)

# Preview first 5 rows
display(df.head())

## 2. Complete Feature List & Data Types

Below is the complete inventory of features in `DEMO_L.xpt`, including their indices, data types, non-null counts, and sample values.

In [ ]:
feature_info = pd.DataFrame({
    'Feature_Index': range(1, num_cols + 1),
    'Feature_Name': df.columns,
    'Data_Type': df.dtypes.values,
    'Non_Null_Count': df.notnull().sum().values,
    'Null_Count': df.isnull().sum().values,
    'Null_Percentage (%)': (df.isnull().sum().values / num_rows * 100).round(2),
    'Unique_Values': [df[col].nunique() for col in df.columns],
    'Sample_Value': [df[col].dropna().iloc[0] if not df[col].dropna().empty else np.nan for col in df.columns]
})

display(feature_info)

## 3. Missing Data Analysis

Understanding missingness patterns is essential for health survey data like NHANES, where certain questions are age-restricted or conditionally asked (skip patterns).

In [ ]:
missing_df = pd.DataFrame({
    'Feature': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values(by='Missing_Percentage (%)', ascending=False).reset_index(drop=True)

print(f"Features with 100% complete data : {(missing_df['Missing_Count'] == 0).sum()} / {num_cols}")
print(f"Features with missing values    : {(missing_df['Missing_Count'] > 0).sum()} / {num_cols}")
print(f"Features with >50% missing data  : {(missing_df['Missing_Percentage (%)'] > 50).sum()} / {num_cols}")

display(missing_df)

# Visualization of Missing Data Percentage
plt.figure(figsize=(12, 6))
sns.barplot(data=missing_df, x='Missing_Percentage (%)', y='Feature', palette='viridis')
plt.title('Percentage of Missing Data by Feature (DEMO_L.xpt)', fontsize=14, fontweight='bold')
plt.xlabel('Missing Percentage (%)')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()

## 4. Descriptive Statistics & Summary Metrics

In [ ]:
desc_stats = df.describe().T
desc_stats['median'] = df.median()
desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
desc_stats['skewness'] = df.skew()
desc_stats['kurtosis'] = df.kurtosis()

# Reorder columns for clarity
desc_stats = desc_stats[['count', 'mean', 'std', 'min', '25%', 'median', '75%', 'max', 'IQR', 'skewness', 'kurtosis']]

print("DESCRIPTIVE STATISTICS SUMMARY:")
display(desc_stats)

## 5. Key Demographic Distribution Breakdown

Exploring foundational NHANES variables:
- `SEQN`: Respondent Sequence Number
- `RIAGENDR`: Gender (1 = Male, 2 = Female)
- `RIDAGEYR`: Age in years at screening
- `RIDRETH1` / `RIDRETH3`: Race/Hispanic origin
- `DMDHHSIZ`: Total number of people in Household
- `INDFMPIR`: Ratio of family income to poverty guidelines

In [ ]:
# Key Variable Labels mapping for visualization clarity
gender_map = {1.0: 'Male', 2.0: 'Female'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Age Distribution
sns.histplot(df['RIDAGEYR'].dropna(), bins=30, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution (RIDAGEYR)', fontweight='bold')
axes[0, 0].set_xlabel('Age (Years)')
axes[0, 0].set_ylabel('Count')

# 2. Gender Breakdown
gender_counts = df['RIAGENDR'].map(gender_map).value_counts()
gender_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90, ax=axes[0, 1], colors=['#66b3ff','#ff9999'])
axes[0, 1].set_title('Gender Distribution (RIAGENDR)', fontweight='bold')
axes[0, 1].set_ylabel('')

# 3. Race / Ethnicity (RIDRETH3)
df['RIDRETH3'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Race/Ethnicity Distribution (RIDRETH3)', fontweight='bold')
axes[1, 0].set_xlabel('Race Category Code')
axes[1, 0].set_ylabel('Count')

# 4. Family Income to Poverty Ratio (INDFMPIR)
sns.histplot(df['INDFMPIR'].dropna(), bins=25, kde=True, ax=axes[1, 1], color='coral')
axes[1, 1].set_title('Poverty Income Ratio (INDFMPIR)', fontweight='bold')
axes[1, 1].set_xlabel('Ratio (Poverty Level <= 1.0)')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 6. Summary & Insights

### Key Takeaways from `DEMO_L.xpt`:
- **Total Sample Size**: **11,933 participants**.
- **Total Features**: **27 demographic & sampling variables**.
- **Participant Age Range**: 0 to 80+ years.
- **Key Features Included**:
  - Unique Identifier: `SEQN`
  - Demographics: Gender (`RIAGENDR`), Age (`RIDAGEYR`, `RIDAGEMN`), Race/Ethnicity (`RIDRETH1`, `RIDRETH3`), Marital Status (`DMDMARTZ`), Education (`DMDEDUC2`)
  - Household: Size (`DMDHHSIZ`), Household reference info (`DMDHRGND`, `DMDHRAGZ`, `DMDHREDZ`, etc.)
  - Socioeconomic: Family Poverty Income Ratio (`INDFMPIR`)
  - Survey Sample Weights: Interview weight (`WTINT2YR`), Examination weight (`WTMEC2YR`), Strata (`SDMVSTRA`), PSU (`SDMVPSU`)